# 学生练习：Online Retail 电商交易分析

> 数据：`06-数据/online_retail_2010_2011.csv`（541,909 行 × 8 列）
> 来源：UCI Online Retail（Chen, 2015），引用见 [02-论文与数据集.md](02-论文与数据集.md)
> 时间：2010-12-01 ~ 2011-12-09

## 怎么用这份 notebook

1. **从上往下**逐格完成，每格里的 `...` 是你要填的代码。
2. 每格下方有【验收点】，写的是"什么算做对了"。
3. 全部写完，运行**最后第 13 格**的自检器，会打印你的得分。
4. 卡住超过 15 分钟再看提示，别直接看教师版。

## 任务地图

| 档 | 任务 | 分值 |
|---|---|---|
| A 必做 | T1 读取体检 · T2 缺失值 · T3 退货 · T4 重复行 · T5 派生列 · T6 清洗 · T7 月度收入 | 55 |
| B 进阶 | T8 国家透视 · T9 Top 商品 · T10 RFM 分层 | 35 |
| C 挑战 | T11 Cohort 留存 · T12 向量化性能 | 10 |

> 三个业务问题：生意怎么样？钱是哪些客户贡献的？货卖得怎么样？
> 做完后，用三句话回答它们，写在最后一格。

## 提交要求（务必先看）

| 项目 | 要求 |
|---|---|
| 组队 | 2 人一组，由**组长**统一提交一份 |
| 文件名 | `组长的学号-组长的姓名.ipynb`（例：`20230101-张三.ipynb`） |
| 提交邮箱 | **20734980@qq.com**，邮件主题写 `学号-姓名-pandas实验` |
| 截止时间 | **上课当天晚 23:00**，逾期不计分 |

提交前请把 notebook **从头运行一遍**，确认最后一格打印出你的得分。

## 0. 环境准备（已写好，直接运行）

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 160)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

DATA = Path('06-数据')

## T1 读取与体检（5 分）

把 CSV 读进来，并回答：多少行、多少列、占多少内存、覆盖什么时间段。

**提示**：`pd.read_csv(..., parse_dates=['InvoiceDate'])`、`df.shape`、`df.memory_usage(deep=True)`。

【验收点】行数 = 541,909；列数 = 8；时间范围 2010-12-01 → 2011-12-09。

In [ ]:
# TODO：读入数据，InvoiceDate 要解析成日期
df = ...

# TODO：形状
n_rows, n_cols = ...

# TODO：内存占用（MB，保留 1 位小数）
mem_mb = ...

# TODO：最早 / 最晚开票时间
date_min = ...
date_max = ...

print(n_rows, n_cols, mem_mb, date_min, date_max)
df.head()

## T2 缺失值（5 分）

统计每列的缺失数与缺失率（百分比，保留 2 位），只保留有缺失的列，按缺失数降序。

**提示**：`df.isna().sum()`、`df.isna().mean()*100`。

**然后想一个问题**：`Customer ID` 缺失意味着什么？该不该删？（写在本格下面的 markdown 里）

【验收点】`miss` 是 DataFrame，索引含 `Customer ID`，其「缺失数」= 135,080。

In [ ]:
# TODO：构造 miss 表，列名必须是 '缺失数' 和 '缺失率%'
miss = ...

miss

**我的回答**：`Customer ID` 缺失代表 ____________________________________，
因为 ____________________________________，所以处理方式是 ____________________________________。

## T3 退货与负金额（10 分）

找出所有取消/退货发票（`InvoiceNo` 以 `C` 开头），回答：
退货多少行、退货金额多少、占正向销售额的百分之几。

**提示**：
- `df.InvoiceNo.str.startswith('C')`
- 退货金额要把负数取正：`-(Quantity * UnitPrice).sum()`
- 正向销售额的分母：`Quantity > 0 且 UnitPrice > 0`

【验收点】`cancel_rows` = 9,288；`return_rate` 与参考答案误差 < 0.05。

In [ ]:
# TODO：筛出取消单
cancel = ...

cancel_rows = ...
cancel_amount = ...   # 取正后的退货金额

# TODO：正向销售额与退货率（百分数）
pos_revenue = ...
return_rate = ...

print(cancel_rows, cancel_amount, pos_revenue, return_rate)

## T4 重复行（5 分）

检查完全重复的行并删除，报告删了多少行。

【验收点】`dup_rows` = 5,268；`len(df_dedup)` = 541,909 − 5,268。

In [ ]:
dup_rows = ...
df_dedup = ...
print(dup_rows, len(df_dedup))

## T5 派生列（10 分）

在 `df` 上新增 5 列：`Revenue`（金额）、`Year`、`Month`、`Weekday`（英文星期名）、`Hour`。

**提示**：`.dt.year / .dt.month / .dt.day_name() / .dt.hour`。

【验收点】5 列都存在；`Revenue` 等于 `Quantity * UnitPrice`；`Hour` 在 0–23 之间。

In [ ]:
# TODO：5 个派生列
df['Revenue'] = ...
df['Year'] = ...
df['Month'] = ...
df['Weekday'] = ...
df['Hour'] = ...

df[['InvoiceDate', 'Revenue', 'Year', 'Month', 'Weekday', 'Hour']].head()

## T6 清洗流水线（10 分）

把下面 5 条规则按顺序写进 `clean_pipeline(raw)`，返回 `(clean, report)`：
`report` 两列：`步骤`、`剩余行数`。

1. 删除完全重复行
2. 只保留 `Quantity > 0`
3. 只保留 `UnitPrice > 0`
4. 剔除特殊编码（`StockCode` 以字母开头，如 `POST`、`M`、`BANK CHARGES`）
5. 剔除 `Customer ID` 缺失的游客订单，并把该列转成 `int`

**提示**：正则 `str.match(r'^[A-Za-z]')` 取反用 `~`。

【验收点】`len(clean)` = 391,150；`clean.Revenue.sum()` 与参考值误差 < 1；`clean.Quantity.min() > 0`。

In [ ]:
def clean_pipeline(raw):
    d = raw.copy()
    steps = [('原始数据', len(d))]

    # TODO ① 去重
    ...
    # TODO ② Quantity > 0
    ...
    # TODO ③ UnitPrice > 0
    ...
    # TODO ④ 剔除字母开头的特殊编码
    ...
    # TODO ⑤ 剔除无客户编号，并转 int
    ...

    return d, pd.DataFrame(steps, columns=['步骤', '剩余行数'])


clean, report = clean_pipeline(df)
report

**想一想**：删掉第 ⑤ 步之后，还能不能回答"公司总销售额是多少"？为什么？

答：____________________________________________________________

## T7 月度与周度收入（10 分）

1. 用 `clean` 按**月**汇总收入、订单数、客户数、商品件数，计算环比。
2. 用 `resample('W')` 算周收入序列。
3. 找出收入最高与最低的月份。

**提示**：`to_period('M')` 让月份可以排序、可以画图。

【验收点】`monthly['收入'].sum()` 与 T6 的有效销售额一致；`best_month` 是 2011-11。

In [ ]:
clean['Month'] = ...

monthly = ...     # 索引是月份，列含 '收入'、'订单数'、'客户数'、'商品件数'、'环比%'
weekly = ...      # Series，索引是周

best_month = ...
worst_month = ...

print(best_month, worst_month)
monthly

## T8 国家透视（10 分）

做 `Country × Month` 的收入透视表，按合计降序，并算出各国收入占比（%）。
`top5_countries` 是占比前 5 的 Series。

**提示**：`pivot_table(index=..., columns=..., values='Revenue', aggfunc='sum', fill_value=0)`。

【验收点】`country_share` 是 Series 且总和 ≈ 100；英国占比约 82.9%。

In [ ]:
country_month = ...
country_share = ...
top5_countries = ...

country_month.head(10)

## T9 Top 商品（10 分）

按 `StockCode` 聚合销售额与销量，分别取 Top10，比较两个榜单。

【验收点】`top_rev` / `top_qty` 各 10 行；销售额第一的编码是 23843。

In [ ]:
by_sku = ...

top_rev = ...
top_qty = ...

print(len(set(top_rev.index) & set(top_qty.index)), '个重叠')

## T10 RFM 分层（15 分）

以数据最后一天 + 1 天为快照日：

- **R** = 距今天数（越小越好，打分时反过来）
- **F** = 不同发票数
- **M** = 总金额

各用 1–5 分打分，再按 R/F/M 是否 ≥4 分成 8 类客户。
输出 `segment`：索引是分层名，列含 `客户数`、`收入合计`、`收入占比%`。

**提示**：`pd.qcut(..., 5, labels=[5,4,3,2,1])`；F/M 先 `rank(method='first')` 再 qcut，避免边界重复报错。

【验收点】`segment['客户数'].sum()` = 客户总数；`segment['收入占比%'].sum()` ≈ 100。

In [ ]:
snapshot = ...

rfm = clean.groupby('Customer ID').agg(
    最近购买=('InvoiceDate', 'max'),
    购买次数=('InvoiceNo', 'nunique'),
    总金额=('Revenue', 'sum'),
)
rfm['R'] = ...
rfm['F'] = ...
rfm['M'] = ...

# TODO 三个 1-5 分列：'R分'、'F分'、'M分'
rfm['R分'] = ...
rfm['F分'] = ...
rfm['M分'] = ...

# TODO 分层列 '客户分层'（8 类，用 np.select 或 map）
rfm['客户分层'] = ...

segment = ...
segment

## T11 Cohort 留存（5 分）

按客户首购月份分组，看每个客户群在首购后第 N 个月还有多少人下单。
输出 `retention`：行=首购月，列=月序，第一列恒为 100。

**提示**：`月序 = 订单年月 - 首购年月`（用 `year*12+month` 换算成整数再相减）；
`pivot_table(..., aggfunc='nunique')` 后除以第 0 列。

【验收点】`retention` 第一列全为 100；行数 ≥ 10。

In [ ]:
first = ...
cohort = ...

tx = ...
tx['月序'] = ...

matrix = ...
retention = ...

retention.iloc[:8, :8]

## T12 向量化性能（5 分）

在 5 万行样本上，分别用 `apply(axis=1)` 和向量化计算金额，比较耗时。

【验收点】`t_fast < t_slow`，且 `t_slow / t_fast > 10`。

In [ ]:
import time

sample = df.head(50_000)

t0 = time.perf_counter()
slow = ...
t_slow = round(time.perf_counter() - t0, 3)

t0 = time.perf_counter()
fast = ...
t_fast = round(time.perf_counter() - t0, 4)

print(t_slow, t_fast, t_slow / t_fast)

## 自检与计分（运行本格）

不用改这一格。它按 T1–T12 的验收点检查你定义的变量，给出得分。
未完成的题会显示 ✗，不会中断。

In [ ]:
CHECKS = [
    ('T1 读取与体检（5 分）', 5,
     lambda: n_rows == 541909 and n_cols == 8 and abs(mem_mb - 160.7) < 3),
    ('T2 缺失值（5 分）', 5,
     lambda: int(miss.loc['Customer ID', '缺失数']) == 135080),
    ('T3 退货分析（10 分）', 10,
     lambda: cancel_rows == 9288 and abs(return_rate - 8.41) < 0.05),
    ('T4 重复行（5 分）', 5,
     lambda: dup_rows == 5268 and len(df_dedup) == 541909 - 5268),
    ('T5 派生列（10 分）', 10,
     lambda: {'Revenue', 'Year', 'Month', 'Weekday', 'Hour'} <= set(df.columns)
             and bool((df['Revenue'] == df['Quantity'] * df['UnitPrice']).all())),
    ('T6 清洗流水线（10 分）', 10,
     lambda: len(clean) == 391150 and abs(clean['Revenue'].sum() - 8737228.0) < 1),
    ('T7 月度收入（10 分）', 10,
     lambda: str(best_month).startswith('2011-11')
             and abs(monthly['收入'].sum() - 8737228.0) < 1),
    ('T8 国家透视（10 分）', 10,
     lambda: len(country_share) == 37 and abs(country_share.sum() - 100) < 0.5),
    ('T9 Top 商品（10 分）', 10,
     lambda: len(top_rev) == 10 and len(top_qty) == 10
             and str(top_rev.index[0]) == '23843'),
    ('T10 RFM 分层（15 分）', 15,
     lambda: abs(segment['客户数'].sum() - 4334) < 1
             and abs(segment['收入占比%'].sum() - 100) < 0.5),
    ('T11 Cohort 留存（5 分）', 5,
     lambda: len(retention) >= 10
             and bool((retention.iloc[:, 0] - 100).abs().max() < 0.01)),
    ('T12 向量化性能（5 分）', 5,
     lambda: t_fast < t_slow and t_slow / max(t_fast, 1e-6) > 10),
]

got = ok_count = 0
for name, pts, fn in CHECKS:
    try:
        ok = bool(fn())
    except Exception:
        ok = False
    got += pts if ok else 0
    ok_count += int(ok)
    print(('✔' if ok else '✗'), f'{name:<20}', f'+{pts}' if ok else '0')

print('-' * 46)
print(f'通过 {ok_count}/{len(CHECKS)} 题，得分 {got}/100')

## 我的三句话结论

用老板听得懂的话，写下三个答案（每句不超过 40 字）：

1. 生意怎么样：______________________________________________
2. 钱是哪些客户贡献的：______________________________________
3. 货卖得怎么样：____________________________________________

## 提交

1. 把整个 notebook **从头运行一遍**，确认最后一格打印出得分；
2. 另存为 `组长的学号-组长的姓名.ipynb`（例：`20230101-张三.ipynb`）；
3. 发送到 **20734980@qq.com**，邮件主题：`学号-姓名-pandas实验`；
4. 截止时间：**上课当天晚 23:00**。

> 每组只交一份，由组长提交。文件名不对或逾期，按未交处理。